In [ ]:
import pandas as pd
Bmarket = pd.read_csv('C:\Users\shiyan\Documents\Bank-additional-full.csv', sep=';')
Bmarket.shape

In [ ]:
# backup first  this cell just run one time
BK_Bmarket= Bmarket.copy()
# drop the highly correlate column 'duration'

import numpy as np
# drop the column duration for all the study
Bmarket.drop('duration', axis = 1, inplace = True)
# change the response variable 
Bmarket.head(1)

In [ ]:
Bmarket.head(2)

In [ ]:
# tree_based methods need to encoding categorical variables
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
#job encode
#le.fit(Bmarket['job'])
#jobcode = le.transform(Bmarket['job']) 

i = 0
for col in ['job','marital','education','default','housing','loan','contact','month','day_of_week','poutcome','y']:
        le.fit(Bmarket[col])
        print (i)
        Bmarket.ix[:,col+str(i)] =le.transform(Bmarket[col])
        i = i + 1
        print (i)
#maritalcode = le.transform(Bmarket['marital']) 
# education encode
#le.fit(Bmarket[''])
#jobcode = le.transform(Bmarket['job']) 
#print (i)
Bmarket.head(2)


In [ ]:
Bmarket.columns

In [ ]:
## reformat the data frame just run once
Bmarket.drop(Bmarket.columns[[1,2,3,4,5,6,7,8,9,13,19]], axis = 1, inplace = True)
Bmarket.head(2)

In [ ]:
# spliting the dataset to train and test
#split the data set into train and test at 80/20
import sklearn
import sklearn.cross_validation
train, test = sklearn.cross_validation.train_test_split(Bmarket, train_size = 0.8)
# varify the train and test
print (len(train.index))
print (len(test.index))
x_train = train.ix[:, train.columns != 'y10']
x_test = test.ix[:, test.columns != 'y10']
y_test = test['y10']
y_train = train['y10']

In [ ]:
# building decision tree model
from sklearn import tree
DT = tree.DecisionTreeClassifier()
DT1 = DT.fit(x_train, y_train)

In [ ]:
print (DT1.score(x_train,y_train))  # score is 0.99344
print (DT1.score(x_test, y_test))   # score is 0.8377  over fitting

In [ ]:
# Since DecisionTreeClassifier does not support prune in python
#output the data for R:
Bmarket.to_csv('BmarketForR.csv', index = False)

In [ ]:
# starting RandomForest model
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=7, )

In [ ]:
# perform model evaluation
# build model evaluation function
from __future__ import print_function
import sklearn.metrics
import time

def evaluate_model(clf):
    """Scores a model using log loss with the created train and test sets."""
    start = time.time()
    clf.fit(x_train, y_train)
    train_loss = sklearn.metrics.log_loss(y_train, clf.predict_proba(x_train))
    test_loss = sklearn.metrics.log_loss(y_test, clf.predict_proba(x_test))
    print("Train score:", train_loss)
    print("Test score:", test_loss)
    print("Total time:", time.time() - start)
    print()
    return test_loss

In [ ]:
# test run on evaluation the model
evaluate_model(rf)

In [ ]:
print(len(x_train.index))
print(len(y_train))

In [ ]:
import sklearn.cross_validation
import numpy as np
import sklearn.learning_curve
import matplotlib.pyplot as plt
%matplotlib inline
max_depths = [1,2,3,4,5,6,7,8] # FILL ME IN
train_scores, test_scores = sklearn.learning_curve.validation_curve(
    RandomForestClassifier(), x_train, y_train, 'max_depth', max_depths)
plt.plot(np.mean(train_scores, axis=1), label='Train')
plt.plot(np.mean(test_scores, axis=1), label='Test')
# based on the plot the max_depth = 2 or 3

In [ ]:
# grid search for the best params
from sklearn.grid_search import GridSearchCV
params = {'class_weight' : [{1:4},{1:3},{1:2}]}
grid = GridSearchCV(RandomForestClassifier(max_depth=4, n_estimators = 1000,max_features = 'log2'), 
                    params, scoring='log_loss')
evaluate_model(grid)

In [ ]:
grid.best_estimator_

In [ ]:
rf_MD = RandomForestClassifier(n_estimators=1000, max_depth=4, n_jobs=1,max_features='log2',class_weight={1:6} )

In [ ]:
rf_MD.fit(x_train,y_train)

In [ ]:
from  sklearn.metrics import classification_report
rf_MD.score(x_train,y_train)  # 10000 trees: 0.89885  100 trees: 0.89875, 100000: 0.89970


In [ ]:
rf.Predict = rf_MD.predict(x_train)
target_names = ['class 0','class 1']
print (classification_report(y_train,rf.Predict, target_names = target_names))


In [ ]:
rf.PredictTest = rf_MD.predict(x_test)
target_names = ['class 0','class 1']
print (classification_report(y_test,rf.PredictTest, target_names = target_names))
# conclusion: 
#the best parameter:n_estimators=1000, max_depth=4, n_jobs=1,max_features='log2',class_weight={1:6} 

In [ ]:
feature_importance = rf.feature_importances_
#make importances relative to max importance
feature_importance = 100.0*(feature_importance / feature_importance.max())
sorted_idx =np.argsort(feature_importance)
pos = np.arange(sorted_idx.shape[0]) + 0.5
plt.barh(pos, feature_importance[sorted_idx], align='center')
plt.yticks(pos,Bmarket.columns[sorted_idx])
plt.xlabel('Relative Importance')
plt.title('Variable Importance')
plt.show()

In [ ]:
Bmarket.head(2)

In [ ]:
# building grading boosting model default n_estimatoers = 100
from sklearn.ensemble import GradientBoostingClassifier

gbt = GradientBoostingClassifier()
evaluate_model(gbt)

In [ ]:
gbt.n_estimators  # default number

In [ ]:
def plot_gbt_learning(gbt):
    test_score = np.empty(len(gbt.estimators_))
    train_score = np.empty(len(gbt.estimators_))
    for i, pred in enumerate(gbt.staged_predict_proba(x_test)):
        test_score[i] = sklearn.metrics.log_loss(y_test, pred)
    for i, pred in enumerate(gbt.staged_predict_proba(x_train)):
        train_score[i] = sklearn.metrics.log_loss(y_train, pred)
    plt.figure(figsize=(8,6))
    plt.plot(np.arange(gbt.n_estimators) + 1, test_score, label='Test') 
    plt.plot(np.arange(gbt.n_estimators) + 1, train_score, label='Train')

In [ ]:
plot_gbt_learning(gbt)

In [ ]:
# This shows overfittin
gbt = GradientBoostingClassifier(n_estimators=150, learning_rate=0.2)
evaluate_model(gbt)     # 500 estimators: 0.253 for Train, 0.27608 for test
plot_gbt_learning(gbt)

In [ ]:
# grid search for the best parameters
params = {'max_depth' : [4,5], 'n_estimators':[60,80, 100],'learning_rate': [0.1,0.15], 
          'subsample': [0.5,1.0]}
grid = GridSearchCV(GradientBoostingClassifier(), params, scoring='log_loss')
evaluate_model(grid)
grid.grid_scores_

In [ ]:
grid.best_estimator_

In [ ]:
params = {'max_depth' : 4, 'n_estimators': 60,'learning_rate': 0.1}
gbt = GradientBoostingClassifier(**params)
sample_weight = map(lambda x: 1 if x == 0 else 7, y_train)
#gbt.fit(x_train,y_train)
gbt.fit(x_train,y_train, sample_weight = sample_weight)
gbt.score(x_train,y_train)



In [ ]:
gbt.Predict = gbt.predict(x_train)
target_names = ['class 0','class 1']
print (classification_report(y_train,gbt.Predict, target_names = target_names))

In [ ]:
gbt.Predict = gbt.predict(x_test)
target_names = ['class 0','class 1']
print (classification_report(y_test,gbt.Predict, target_names = target_names))

In [ ]:
feature_importance = gbt.feature_importances_
#make importances relative to max importance
feature_importance = 100.0*(feature_importance / feature_importance.max())
sorted_idx =np.argsort(feature_importance)
pos = np.arange(sorted_idx.shape[0]) + 0.5
plt.barh(pos, feature_importance[sorted_idx], align='center')
plt.yticks(pos,Bmarket.columns[sorted_idx])
plt.xlabel('Relative Importance')
plt.title('Variable Importance')
plt.show()

In [ ]:
# Support Vector Classification
from sklearn.svm import SVC
clf=SVC()
clf.fit(x_train,y_train)




In [ ]:
SVC(C=1.0, class_weight = {1:8}, coef0=0.0,
   decision_function_shape = None,degree = 3, gamma = 'auto', kernel = 'poly',
   max_iter = -1, probability = True, random_state= None, shrinking=True,
   tol = 0.01, verbose = False)

In [ ]:
# predcit on train set
PredictFromSVC = clf.predict(x_train)
PredictOnTest = clf.predict(x_test)


In [ ]:
target_names = ['class 0','class 1']
print (classification_report(y_train,PredictFromSVC, target_names = target_names))
print (classification_report(y_test,PredictOnTest, target_names = target_names))

In [ ]:
# the above results seems overfitting
params = {'C' : [0.5, 1.0, 1.5], 'kernel':['linear','poly','rbf','sigmoid'],'degree': [2,3], 
          'coef0': [0,1.0], 'class_weight' :[{1:9},{1:8}]}
grid = GridSearchCV(SVC(probability = True), params, scoring='log_loss')
evaluate_model(grid)
grid.grid_scores_